# YaRN 

In [1]:
# 导库
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
# 先搓一个RoPE
# 手搓频率生成
def inv_freq(dim,base=10000):
    assert dim % 2 == 0, "dim must be even"
    idx = torch.arange(0,dim,2).float()  # [0, 2, 4, ..., dim-2]
    inv_freqs = 1.0 /(base**(idx/dim))
    return inv_freqs

# 手搓cos sin
def get_sin_cos(seq,inv_freqs):
    # seq = [B,T]
    # inv_freq: [dim//2]
    freqs = torch.einsum("bt,d->btd", seq.float(), inv_freqs)  # [B, T, dim//2]
    # 对齐维度并且对半法
    emb = torch.cat([freqs, freqs], dim=-1)  # [B, T, dim]
    # cos和sin都是[B, T, dim]
    cos = emb.cos()
    sin = emb.sin()
    return cos, sin

def apply_rope(x, cos, sin):
    # x: [B, T, dim]
    # cos, sin: [B, T, dim]
    # 先对半法
    x1, x2 = x.chunk(2, dim=-1)  # [B, T, dim//2], [B, T, dim//2]
    cos1, cos2 = cos.chunk(2, dim=-1)  # [B, T, dim//2], [B, T, dim//2]
    sin1, sin2 = sin.chunk(2, dim=-1)  # [B, T, dim//2], [B, T, dim//2]
    # 旋转,x1,x2主要描述是向量的前半部分和后半部分
    x1_new = x1 * cos1 - x2 * sin1
    x2_new = x2 * cos2 + x1 * sin2
    return torch.cat([x1_new,x2_new],dim=-1)


In [ ]:
# 进行yarn改造
def yarn_inv_freq(dim,base=10000,scale=4):
    # scale 就是扩展倍数 (max_position / original_max_position)
    idx = torch.arange(0, dim, 2).float()
    
    # YaRN 的核心：不仅仅是除以 scale，而是对频率分布进行平滑处理
    # 这里的公式通常涉及对高频和低频维度的不同处理（NTK-aware 的进化版）
    inv_freqs = 1.0 / (base ** (idx / dim))
    
    # 基础版本的 YaRN 会直接对 inv_freqs 进行缩放调整
    # 但更高级的实现会根据维度 idx 计算掩码，区分哪些维度需要插值
    return inv_freqs / scale

def yarn_get_sin_cos(seq, inv_freqs,temp=1):
    # 1. 正常的 einsum 计算相位
    freqs = torch.einsum("bt,d->btd", seq.float(), inv_freqs)
    
    # 2. 计算 YaRN 的修正系数 m (Attention Scaling)
    # 这是一个经验公式，用于保持注意力分布的熵
    m = 0.1 * torch.log(torch.tensor(temp)) + 1.0 if temp > 1 else 1.0
    
    emb = torch.cat([freqs, freqs], dim=-1)
    
    # 3. 计算 cos/sin 时，结果要乘以修正系数 m
    # 注意：有的实现是直接给结果乘 m，有的是对旋转后的向量乘 m
    cos = emb.cos() * m
    sin = emb.sin() * m
    return cos, sin